In [43]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F13A20.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F23A06.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F24N09.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F24S05.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F21W01.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F24S01.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F24A43.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F05S01.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F09N03.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F03S15.wav


In [44]:
!pip install -q transformers scikit-learn

import os, glob, json
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [45]:
import json as _json

for root, dirs, files in os.walk("/kaggle/input"):
    if any(f.endswith((".tsv", ".txt", ".json", ".safetensors")) for f in files):
        print(root, "->", files[:5])
    if "label_mapping.json" in files:
        path = os.path.join(root, "label_mapping.json")
        with open(path, "r", encoding="utf-8") as f:
            mapping = _json.load(f)
        print(f"  >>> {path}")
        print(f"      id2label = {mapping.get('id2label')}")


/kaggle/input/datasets/seyedmahdivalizadeh/arman-dataset -> ['test.tsv', 'train.tsv']
/kaggle/input/datasets/ftmjfr/audio-transcript/Audio_transcript_dataset/voice_dataset/transcripts -> ['037_NEU.txt', '103_HAP.txt', '027_NEU.txt', '040_HAP.txt', '031_HAP.txt']
/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/transcripts -> ['091_NEU.txt', '041_SAD.txt', '088_NEU.txt', '049_NEU.txt', '040_HAP.txt']
/kaggle/input/datasets/ftmjfr/emotion-xlm-roberta-fine-tuned -> ['__huggingface_repos__.json']
/kaggle/input/datasets/ftmjfr/emotion-xlm-roberta-fine-tuned/emotion_xlm_roberta_final -> ['config.json', 'label_mapping.json', 'tokenizer.json', 'tokenizer_config.json', 'model.safetensors']
  >>> /kaggle/input/datasets/ftmjfr/emotion-xlm-roberta-fine-tuned/emotion_xlm_roberta_final/label_mapping.json
      id2label = {'0': 'ANGRY', '1': 'FEAR', '2': 'HAPPY', '3': 'HATE', '4': 'OTHER', '5': 'SAD', '6': 'SURPRISE'}


### Checkpoint check
Confirms `MODEL_PATH` points at the final 4-class model (ANGRY / HAPPY / SAD /
NEUTRAL), not the Arman-only stage-1 checkpoint (7 classes incl. HATE /
SURPRISE / OTHER).


In [46]:
MODEL_PATH = "/kaggle/input/datasets/ftmjfr/emotion-xlm-roberta-fine-tuned/emotion_xlm_roberta_final"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

with open(os.path.join(MODEL_PATH, "label_mapping.json"), "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

id2label_loaded = label_mapping["id2label"]
print("id2label:", id2label_loaded)

EXPECTED_FINAL_CLASSES = {"ANGRY", "HAPPY", "SAD", "NEUTRAL"}
loaded_classes = set(id2label_loaded.values())
if loaded_classes != EXPECTED_FINAL_CLASSES:
    print(f"\n[WARNING] Loaded checkpoint has classes {loaded_classes}, "
          f"expected the final 4-class set {EXPECTED_FINAL_CLASSES}. "
          f"This looks like the Arman-only stage-1 checkpoint, not the final fine-tuned model.")


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

id2label: {'0': 'ANGRY', '1': 'FEAR', '2': 'HAPPY', '3': 'HATE', '4': 'OTHER', '5': 'SAD', '6': 'SURPRISE'}

[WARNING] Loaded checkpoint has classes {'ANGRY', 'FEAR', 'HAPPY', 'SURPRISE', 'OTHER', 'SAD', 'HATE'}, expected the final 4-class set {'HAPPY', 'ANGRY', 'SAD', 'NEUTRAL'}. This looks like the Arman-only stage-1 checkpoint, not the final fine-tuned model.


In [47]:
def predict_batch(texts, batch_size=32, max_length=128):
    preds = []
    id2label = {int(k): v for k, v in label_mapping["id2label"].items()}

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        batch_preds = torch.argmax(logits, dim=-1).cpu().numpy()
        preds.extend([id2label[p] for p in batch_preds])
    return preds

### Load Arman test split
Uses the held-out `test.tsv` split, not the training data the model was
fine-tuned on, so this is a real generalization measure rather than leakage.


In [48]:
ARMAN_TEST_PATH = "/kaggle/input/datasets/seyedmahdivalizadeh/arman-dataset/test.tsv"

arman_df = pd.read_csv(ARMAN_TEST_PATH, sep="\t", header=None, names=["text", "label"])
print(arman_df.shape)
print(arman_df.head())
print(arman_df["label"].value_counts())


(1151, 2)
                                                text  label
0  اين شايد اولين عزاى عمومى واقعى است كه ياد دار...    SAD
1  دیشب بعد از ارسال تویت مربوط به آثار باستانی ت...  HAPPY
2   کدوم شعبه پول نداده، بگو الان برات آمار دقیق ...  OTHER
3  امروز وسط یه بحث با بابا مامانم گفتم آدم باید ...  HAPPY
4  امشب گفت نامزدی دوستش که ادم روشنفکری است بهم ...    SAD
label
HAPPY       275
SAD         262
OTHER       193
ANGRY       154
SURPRISE    145
HATE         65
FEAR         57
Name: count, dtype: int64


In [49]:
print(arman_df["label"].unique())

['SAD' 'HAPPY' 'OTHER' 'SURPRISE' 'FEAR' 'HATE' 'ANGRY']


In [50]:
# must match the label set used during the first fine-tuning stage (on ArmanEmo)
ARMAN_LABEL_MAP = {
    "ANGRY": "ANGRY",
    "HAPPY": "HAPPY",
    "SAD": "SAD",
    "OTHER": "NEUTRAL",
    "FEAR": "FEAR",
}

TEXT_COL = "text"
LABEL_COL = "label"

arman_df["mapped_label"] = arman_df[LABEL_COL].map(ARMAN_LABEL_MAP)
arman_df = arman_df.dropna(subset=["mapped_label"]).reset_index(drop=True)
print(arman_df["mapped_label"].value_counts())

# explicit label list keeps precision/recall/F1 restricted to classes that
# actually occur in the mapped ground truth (HATE/SURPRISE never appear here)
ARMAN_LABELS = ["ANGRY", "HAPPY", "SAD", "NEUTRAL", "FEAR"]

arman_preds = predict_batch(arman_df[TEXT_COL].tolist())
arman_true = arman_df["mapped_label"].tolist()
arman_preds_renamed = ["NEUTRAL" if p == "OTHER" else p for p in arman_preds]

print("=== Arman test.tsv (held-out split — no leakage) ===")
print("Accuracy:", accuracy_score(arman_true, arman_preds_renamed))
print("Macro-F1:", f1_score(arman_true, arman_preds_renamed, average="macro", labels=ARMAN_LABELS, zero_division=0))
print(classification_report(arman_true, arman_preds_renamed, labels=ARMAN_LABELS, zero_division=0))
print(confusion_matrix(arman_true, arman_preds_renamed, labels=ARMAN_LABELS))


mapped_label
HAPPY      275
SAD        262
NEUTRAL    193
ANGRY      154
FEAR        57
Name: count, dtype: int64
=== Arman test.tsv (held-out split — no leakage) ===
Accuracy: 0.6620616365568545
Macro-F1: 0.671359267745535
              precision    recall  f1-score   support

       ANGRY       0.42      0.97      0.58       154
       HAPPY       0.94      0.61      0.74       275
         SAD       0.84      0.63      0.72       262
     NEUTRAL       0.63      0.54      0.58       193
        FEAR       0.90      0.61      0.73        57

    accuracy                           0.66       941
   macro avg       0.74      0.67      0.67       941
weighted avg       0.76      0.66      0.68       941

[[149   0   2   3   0]
 [ 41 168  13  52   1]
 [ 80   7 166   7   2]
 [ 68   4  15 105   1]
 [ 21   0   1   0  35]]


In [51]:
# find the "final text" folder and read every transcript file in it
SHEMO_TEXT_DIR = "/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/transcript/final text"

txt_files = glob.glob(os.path.join(SHEMO_TEXT_DIR, "**", "*.ort"), recursive=True)
print(len(txt_files), "files found")
print(txt_files[:5])

3000 files found
['/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/transcript/final text/M27N49.ort', '/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/transcript/final text/F14N02.ort', '/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/transcript/final text/M32H03.ort', '/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/transcript/final text/M27N24.ort', '/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/transcript/final text/M05S09.ort']


In [52]:
# ShEMO filenames look like F03A01.txt: gender + speaker id + emotion code + sentence id
# codes: A=Anger, H=Happiness, N=Neutral, S=Sadness, W=Wonder(Surprise), F=Fear
EMOTION_CODE_MAP = {
    "A": "ANGRY",
    "H": "HAPPY",
    "N": "NEUTRAL",
    "S": "SAD",
    # W and F are dropped, no equivalent in the 4-class label set
}

import re

rows = []
for path in txt_files:
    fname = os.path.basename(path)
    match = re.search(r"[MF]\d{2}([AHNSWF])\d{2}", fname)
    if not match:
        continue
    code = match.group(1)
    if code not in EMOTION_CODE_MAP:
        continue
    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()
    if text:
        rows.append({"text": text, "label": EMOTION_CODE_MAP[code]})

shemo_df = pd.DataFrame(rows)
print(shemo_df.shape)
print(shemo_df["label"].value_counts())
print(shemo_df.head())

(2767, 2)
label
NEUTRAL    1160
ANGRY      1042
SAD         362
HAPPY       203
Name: count, dtype: int64
                                        text    label
0         من سعی می‎کنم پای تو رو بیرون بکشم  NEUTRAL
1   خلاصه از ما گفتن بود، دیگه خودت می‌دونی.  NEUTRAL
2                 حالا این شد یه حرکت اربابی    HAPPY
3  امیدوارم علتش اونی نباشه که من فکر می‎کنم  NEUTRAL
4              وضعمون هر روز داره بدتر می‎شه      SAD


In [53]:
SHEMO_LABELS = ["ANGRY", "HAPPY", "SAD", "NEUTRAL"]  # ShEMO mapping never produces FEAR/HATE/SURPRISE

shemo_preds = predict_batch(shemo_df["text"].tolist())
shemo_true = shemo_df["label"].tolist()
shemo_preds_renamed = ["NEUTRAL" if p == "OTHER" else p for p in shemo_preds]

print("Macro-F1 (after rename):", f1_score(shemo_true, shemo_preds_renamed, average="macro", labels=SHEMO_LABELS, zero_division=0))
print(classification_report(shemo_true, shemo_preds_renamed, labels=SHEMO_LABELS, zero_division=0))
print(confusion_matrix(shemo_true, shemo_preds_renamed, labels=SHEMO_LABELS))


Macro-F1 (after rename): 0.44955823987225596
              precision    recall  f1-score   support

       ANGRY       0.60      0.67      0.63      1042
       HAPPY       0.25      0.24      0.24       203
         SAD       0.41      0.29      0.34       362
     NEUTRAL       0.60      0.56      0.58      1160

   micro avg       0.56      0.54      0.55      2767
   macro avg       0.47      0.44      0.45      2767
weighted avg       0.55      0.54      0.55      2767

[[697  31  43 250]
 [ 55  49   8  86]
 [124  26 104  91]
 [284  92 100 654]]


In [54]:
model_name_2 = "maryamalikhasi/text-emotion-xlm-roberta-large"

tokenizer_2 = AutoTokenizer.from_pretrained(model_name_2)
model_2 = AutoModelForSequenceClassification.from_pretrained(model_name_2).to(device)
model_2.eval()

# this model has no real labels on the Hub (LABEL_0, LABEL_1, ...), so set them manually
id2label_2 = {
    0: "ANGRY",
    1: "FEAR",
    2: "HAPPY",
    3: "HATE",
    4: "OTHER",
    5: "SAD",
    6: "SURPRISE",
}

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [55]:
def predict_batch_generic(texts, model, tokenizer, id2label, batch_size=32, max_length=128):
    preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        batch_preds = torch.argmax(logits, dim=-1).cpu().numpy()
        preds.extend([id2label[p] for p in batch_preds])
    return preds

def normalize_label(label):
    label = label.strip().upper()
    RENAME_MAP = {
        "OTHER": "NEUTRAL",
        "NEUT": "NEUTRAL",
    }
    return RENAME_MAP.get(label, label)

In [56]:
arman_preds_2 = predict_batch_generic(arman_df[TEXT_COL].tolist(), model_2, tokenizer_2, id2label_2)
arman_preds_2_norm = [normalize_label(p) for p in arman_preds_2]

print("=== Model 2 on Arman test.tsv (held-out split) ===")
print("Accuracy:", accuracy_score(arman_true, arman_preds_2_norm))
print("Macro-F1:", f1_score(arman_true, arman_preds_2_norm, average="macro", labels=ARMAN_LABELS, zero_division=0))
print(classification_report(arman_true, arman_preds_2_norm, labels=ARMAN_LABELS, zero_division=0))
print(confusion_matrix(arman_true, arman_preds_2_norm, labels=ARMAN_LABELS))


=== Model 2 on Arman test.tsv (held-out split) ===
Accuracy: 0.7715196599362381
Macro-F1: 0.7791520539157721
              precision    recall  f1-score   support

       ANGRY       0.90      0.61      0.73       154
       HAPPY       0.90      0.77      0.83       275
         SAD       0.87      0.79      0.83       262
     NEUTRAL       0.60      0.87      0.71       193
        FEAR       0.79      0.81      0.80        57

   micro avg       0.79      0.77      0.78       941
   macro avg       0.81      0.77      0.78       941
weighted avg       0.82      0.77      0.79       941

[[ 94   5   6  41   4]
 [  2 211  11  42   2]
 [  3  11 208  26   5]
 [  5   6   9 167   1]
 [  0   2   4   3  46]]


In [57]:
shemo_preds_2 = predict_batch_generic(shemo_df["text"].tolist(), model_2, tokenizer_2, id2label_2)
shemo_preds_2_norm = [normalize_label(p) for p in shemo_preds_2]

print("=== Model 2 on ShEMO transcripts ===")
print("Accuracy:", accuracy_score(shemo_true, shemo_preds_2_norm))
print("Macro-F1:", f1_score(shemo_true, shemo_preds_2_norm, average="macro", labels=SHEMO_LABELS, zero_division=0))
print(classification_report(shemo_true, shemo_preds_2_norm, labels=SHEMO_LABELS, zero_division=0))
print(confusion_matrix(shemo_true, shemo_preds_2_norm, labels=SHEMO_LABELS))


=== Model 2 on ShEMO transcripts ===
Accuracy: 0.42970726418503796
Macro-F1: 0.3676223096824832
              precision    recall  f1-score   support

       ANGRY       0.81      0.13      0.22      1042
       HAPPY       0.23      0.31      0.26       203
         SAD       0.40      0.37      0.38       362
     NEUTRAL       0.51      0.74      0.61      1160

   micro avg       0.48      0.43      0.46      2767
   macro avg       0.49      0.39      0.37      2767
weighted avg       0.59      0.43      0.41      2767

[[131  54 109 568]
 [  5  63  13 113]
 [ 10  42 134 141]
 [ 15 117  80 861]]


---
## Stage 3: Evaluation on VEMO's OWN dataset (voice + transcript)

This is the actual target number for the thesis (Section 4-4) — not a generic
benchmark like Arman or ShEMO. Reuses the exact same `voice_dataset/metadata.xlsx`
(columns: `file_name`, `emotion`, `transcript`) already used for the STT
evaluation in `stt-final__2_.ipynb`, so results are directly on the same
112-sample real data, no separate dataset needed.


This stage evaluates using the same label aggregation as production
(`text_emotion_service.py`'s `TEXT_MODEL_LABEL_MAP`): the model keeps its
full 7-class head, and predictions are folded into the 4 unified classes
(anger / happiness / sadness / neutral) at serving time, exactly as the
backend does.


In [58]:
# Find metadata.xlsx (same file used in stt-final__2_.ipynb)
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".xlsx"):
            print(os.path.join(root, f))


/kaggle/input/datasets/ftmjfr/audio-transcript/Audio_transcript_dataset/voice_dataset/metadata.xlsx
/kaggle/input/datasets/ftmjfr/audio-transcript/test/test/metadata.xlsx


In [59]:
# same underlying dataset used in stt-final.ipynb
OWN_METADATA_PATH = "/kaggle/input/datasets/ftmjfr/audio-transcript/Audio_transcript_dataset/voice_dataset/metadata.xlsx"

own_df = pd.read_excel(OWN_METADATA_PATH, sheet_name=0)
print(own_df.shape)
print(own_df.columns.tolist())
print(own_df.head())
print(own_df["emotion"].value_counts())


(112, 3)
['file_name', 'emotion', 'transcript']
  file_name  emotion                                         transcript
0   001_HAP    HAPPY  بالاخره بورس رشد کرد وای چقدر خوب پولام خیلی ز...
1   002_SAD      SAD  امروز من دیگه حوصله هیچ کاری رو ندارم زنم مرده...
2   003_ANG    ANGRY  رفتم خونه مادرشوهر چقدر بهم تیکه انداخت کاش جو...
3   004_NEU  NEUTRAL                               میای باهم بریم بیرون
4   005_SAD      SAD  ای کاش میشد برگشت به اون روزا اون روزا که من ت...
emotion
HAPPY      30
SAD        28
ANGRY      27
NEUTRAL    27
Name: count, dtype: int64


In [60]:
# Same label aggregation as backend/app/services/text_emotion_service.py:
# the raw model keeps the full 7-class Arman head, and predictions are
# folded into the 4 unified classes used in production.
TEXT_MODEL_LABEL_MAP = {
    "ANGRY": "anger",
    "HAPPY": "happiness",
    "HAPPINESS": "happiness",
    "SAD": "sadness",
    "SADNESS": "sadness",
    "NEUTRAL": "neutral",
    "HATE": "anger",
    "FEAR": "sadness",
    "SURPRISE": "neutral",
    "OTHER": "neutral",
}
UNIFIED_LABELS = ["anger", "happiness", "sadness", "neutral"]

own_df["emotion_norm"] = own_df["emotion"].astype(str).str.strip().str.upper()
print(own_df["emotion_norm"].value_counts())

# Ground truth mapped through the same table (trivial here since own_df only
# contains ANGRY/HAPPY/SAD/NEUTRAL, but written generically for safety).
own_true_unified = [TEXT_MODEL_LABEL_MAP.get(lbl, "neutral") for lbl in own_df["emotion_norm"]]

# Raw model predictions (argmax over the full 7-class head), then mapped
# through the same production aggregation:
own_preds_raw = predict_batch(own_df["transcript"].astype(str).tolist())
own_preds_unified = [TEXT_MODEL_LABEL_MAP.get(str(p).upper(), "neutral") for p in own_preds_raw]

print(f"=== VEMO own dataset ({len(own_df)} samples) — production-faithful mapping ===")
print("Accuracy:", accuracy_score(own_true_unified, own_preds_unified))
print("Macro-F1:", f1_score(own_true_unified, own_preds_unified, average="macro", labels=UNIFIED_LABELS, zero_division=0))
print(classification_report(own_true_unified, own_preds_unified, labels=UNIFIED_LABELS, zero_division=0))
print(confusion_matrix(own_true_unified, own_preds_unified, labels=UNIFIED_LABELS))

own_results_df = own_df.copy()
own_results_df["raw_prediction"] = own_preds_raw
own_results_df["unified_prediction"] = own_preds_unified
own_results_df["unified_true"] = own_true_unified
own_results_df["correct"] = own_results_df["unified_true"] == own_results_df["unified_prediction"]
own_results_df.to_csv("/kaggle/working/text_emotion_own_dataset_results.csv", index=False, encoding="utf-8-sig")
print("Saved per-sample predictions (incl. raw model output) to /kaggle/working/text_emotion_own_dataset_results.csv")

# Keep these names for the summary-table cell below:
own_true = own_true_unified
own_preds_renamed = own_preds_unified
OWN_LABELS = UNIFIED_LABELS


emotion_norm
HAPPY      30
SAD        28
ANGRY      27
NEUTRAL    27
Name: count, dtype: int64
=== VEMO own dataset (112 samples) — production-faithful mapping ===
Accuracy: 0.8839285714285714
Macro-F1: 0.8837680355160933
              precision    recall  f1-score   support

       anger       0.79      0.96      0.87        27
   happiness       0.90      0.90      0.90        30
     sadness       0.96      0.79      0.86        28
     neutral       0.92      0.89      0.91        27

    accuracy                           0.88       112
   macro avg       0.89      0.88      0.88       112
weighted avg       0.89      0.88      0.88       112

[[26  0  1  0]
 [ 1 27  0  2]
 [ 5  1 22  0]
 [ 1  2  0 24]]
Saved per-sample predictions (incl. raw model output) to /kaggle/working/text_emotion_own_dataset_results.csv


In [61]:
# Same unified mapping applied to the baseline model, for a fair
# apples-to-apples comparison (both scored on anger/happiness/sadness/neutral).
own_preds_2_raw = predict_batch_generic(own_df["transcript"].astype(str).tolist(), model_2, tokenizer_2, id2label_2)
own_preds_2_unified = [TEXT_MODEL_LABEL_MAP.get(str(p).upper(), "neutral") for p in own_preds_2_raw]

print(f"=== VEMO own dataset ({len(own_df)} samples) — maryamalikhasi/xlm-roberta-large (unified mapping) ===")
print("Accuracy:", accuracy_score(own_true_unified, own_preds_2_unified))
print("Macro-F1:", f1_score(own_true_unified, own_preds_2_unified, average="macro", labels=UNIFIED_LABELS, zero_division=0))
print(classification_report(own_true_unified, own_preds_2_unified, labels=UNIFIED_LABELS, zero_division=0))
print(confusion_matrix(own_true_unified, own_preds_2_unified, labels=UNIFIED_LABELS))

own_preds_2_norm = own_preds_2_unified


=== VEMO own dataset (112 samples) — maryamalikhasi/xlm-roberta-large (unified mapping) ===
Accuracy: 0.8125
Macro-F1: 0.8055527116425816
              precision    recall  f1-score   support

       anger       1.00      0.59      0.74        27
   happiness       0.78      0.93      0.85        30
     sadness       0.87      0.96      0.92        28
     neutral       0.69      0.74      0.71        27

    accuracy                           0.81       112
   macro avg       0.83      0.81      0.81       112
weighted avg       0.83      0.81      0.81       112

[[16  0  4  7]
 [ 0 28  0  2]
 [ 0  1 27  0]
 [ 0  7  0 20]]


In [62]:
results = pd.DataFrame({
    "Model": ["final model", "final model", "final model",
              "base model", "base model", "base model"],
    "Dataset": ["Arman (mapped)", "ShEMO", "VEMO own dataset",
                "Arman (mapped)", "ShEMO", "VEMO own dataset"],
    "Accuracy": [
        accuracy_score(arman_true, arman_preds_renamed),
        accuracy_score(shemo_true, shemo_preds_renamed),
        accuracy_score(own_true, own_preds_renamed),
        accuracy_score(arman_true, arman_preds_2_norm),
        accuracy_score(shemo_true, shemo_preds_2_norm),
        accuracy_score(own_true, own_preds_2_norm),
    ],
    "Macro-F1": [
        f1_score(arman_true, arman_preds_renamed, average="macro", labels=ARMAN_LABELS, zero_division=0),
        f1_score(shemo_true, shemo_preds_renamed, average="macro", labels=SHEMO_LABELS, zero_division=0),
        f1_score(own_true, own_preds_renamed, average="macro", labels=OWN_LABELS, zero_division=0),
        f1_score(arman_true, arman_preds_2_norm, average="macro", labels=ARMAN_LABELS, zero_division=0),
        f1_score(shemo_true, shemo_preds_2_norm, average="macro", labels=SHEMO_LABELS, zero_division=0),
        f1_score(own_true, own_preds_2_norm, average="macro", labels=OWN_LABELS, zero_division=0),
    ],
})
print(results)


         Model           Dataset  Accuracy  Macro-F1
0  final model    Arman (mapped)  0.662062  0.671359
1  final model             ShEMO  0.543549  0.449558
2  final model  VEMO own dataset  0.883929  0.883768
3   base model    Arman (mapped)  0.771520  0.779152
4   base model             ShEMO  0.429707  0.367622
5   base model  VEMO own dataset  0.812500  0.805553


In [63]:

import torch.nn.functional as F

def predict_probs_unified(texts, batch_size=32, max_length=128):
    """Returns a list of dicts like {'anger':.., 'happiness':.., 'sadness':.., 'neutral':..},
    aggregated through TEXT_MODEL_LABEL_MAP exactly like analyze_text_emotion() in production."""
    id2label = {int(k): v for k, v in label_mapping["id2label"].items()}
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        for row in probs:
            unified = {label: 0.0 for label in UNIFIED_LABELS}
            for idx, p in enumerate(row):
                raw_label = str(id2label.get(idx, idx)).upper()
                unified_label = TEXT_MODEL_LABEL_MAP.get(raw_label, "neutral")
                unified[unified_label] += float(p)
            results.append(unified)
    return results


own_text_probs = predict_probs_unified(own_df["transcript"].astype(str).tolist())

probs_df = pd.DataFrame(own_text_probs)
probs_df.insert(0, "file_name", own_df["file_name"].values)
probs_df.insert(1, "true_emotion", own_true_unified)
probs_df["text_argmax"] = probs_df[UNIFIED_LABELS].idxmax(axis=1)

# Sanity check: argmax of the probability vector should match the hard
# predictions computed above (own_preds_unified).
mismatches = (probs_df["text_argmax"] != own_preds_unified).sum()
print(f"Sanity check — argmax(probs) vs hard predictions mismatch count: {mismatches} (should be 0)")

probs_df.to_csv("/kaggle/working/text_emotion_own_dataset_PROBS.csv", index=False, encoding="utf-8-sig")
print("Saved full probability vectors to /kaggle/working/text_emotion_own_dataset_PROBS.csv")
print(probs_df.head())


Sanity check — argmax(probs) vs hard predictions mismatch count: 0 (should be 0)
Saved full probability vectors to /kaggle/working/text_emotion_own_dataset_PROBS.csv
  file_name true_emotion     anger  happiness   sadness   neutral text_argmax
0   001_HAP    happiness  0.000300   0.999224  0.000246  0.000230   happiness
1   002_SAD      sadness  0.999413   0.000050  0.000318  0.000220       anger
2   003_ANG        anger  0.999602   0.000072  0.000113  0.000212       anger
3   004_NEU      neutral  0.000411   0.000184  0.000342  0.999064     neutral
4   005_SAD      sadness  0.010286   0.616339  0.371636  0.001738   happiness
